# Workbook 1 · Your first variant: change the mission and read what happened

The tutorials explain how the model works. This workbook is about **using** it: you change something, size the aircraft, and read the answer like a
designer. Every exercise has the same five steps:

1. **Start** from the baseline (the cached near-reference of tutorial 19: every model at the reference setting, Scholz aerodynamics).
2. **Change** one input: a requirement (`HaloRequirements`) or an assumption (`HaloAssumptions`), with `replace(...)`.
3. **Size** it: `size(label, requirements=..., assumptions=...)`, one warm-started solve, about 20-40 s.
4. **Compare**: `compare(base, variant)` for the key numbers, `weight_changes(...)` for where the mass went.
5. **Explain**: `binding_changes(...)` says which requirements started or stopped sizing the aircraft. That is the *why*.

`sizing_workbench.py` (in `tutorials/`) holds these helpers; each is about ten lines, so read it once.

**Run time:** about 3 minutes (four solves). The first run solves the baseline if the cache is empty (about a minute).

In [1]:
import sys
from pathlib import Path
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(repo_root / "tutorials"))
from sizing_workbench import *
from dataclasses import replace
base = baseline_variant()
compare(base)

                                            baseline
take-off weight [lb]                          15,134
empty weight [lb]                             11,087
payload [lb]                                   1,984
fuel [lb]                                      2,063
battery [kWh]                                   69.5
wing area [m2]                                 23.24
span [m]                                       11.93
rotor radius [m]                               4.825
disk loading [kg/m2]                            46.9
cruise speed [kt]                                161
cruise L/D                                      9.05
motor rating per lane [kW]                       528
generator rating [kW]                            945
cost per mission [USD]                         3,342
machine units (motor, gen)                    (2, 3)


## 1. More payload

The baseline carries 900 kg. What does 1,100 kg cost?

In [2]:
heavy = size("payload 1,100 kg", requirements=replace(base.requirements, mass_payload_kg=1100.0))
compare(base, heavy)

[payload 1,100 kg: sized in 27 s from 'baseline': 16,252 lb]
                                            baseline        payload 1,100 kg
take-off weight [lb]                          15,134          16,252 (+7.4%)
empty weight [lb]                             11,087          11,682 (+5.4%)
payload [lb]                                   1,984          2,425 (+22.2%)
fuel [lb]                                      2,063           2,145 (+4.0%)
battery [kWh]                                   69.5           81.3 (+17.0%)
wing area [m2]                                 23.24           24.62 (+5.9%)
span [m]                                       11.93           12.27 (+2.9%)
rotor radius [m]                               4.825           4.999 (+3.6%)
disk loading [kg/m2]                            46.9            46.9 (+0.0%)
cruise speed [kt]                                161             162 (+0.6%)
cruise L/D                                      9.05            9.19 (+1.6%)
motor rating pe

Read the table top to bottom: 200 kg (441 lb) more payload costs far more than 441 lb of take-off weight. Work out the **growth factor** from the table:

In [3]:
growth = (heavy.result.mass_takeoff_kg - base.result.mass_takeoff_kg) / 200.0
print(f"growth factor: {growth:.2f} kg of take-off mass per kg of payload")
check("payload growth costs more than its own weight", growth > 1)

growth factor: 2.54 kg of take-off mass per kg of payload
PASS  payload growth costs more than its own weight


Where did the extra weight go?

In [4]:
weight_changes(base, heavy)

group                                     baseline  payload 1,100       change
payload                                    1,984 lb         2,425 lb      +441 lb
powertrain: battery                        1,041 lb         1,217 lb      +176 lb
powertrain: propulsor                      1,482 lb         1,623 lb      +141 lb
fuel                                       2,063 lb         2,145 lb       +83 lb
wing                                         897 lb           962 lb       +66 lb
systems                                      895 lb           958 lb       +63 lb
powertrain: gearbox                          642 lb           701 lb       +59 lb
landing_gear                                 568 lb           599 lb       +31 lb
powertrain: heat_exchanger                   278 lb           306 lb       +28 lb
fuselage                                   1,217 lb         1,234 lb       +17 lb
take-off weight                           15,134 lb        16,252 lb    +1,118 lb


And **why** those parts? Ask what is binding now that was not before:

In [5]:
binding_changes(base, heavy)

now binding (new):
    -
no longer binding:
    -
still binding:
    bus-out hover: motor torque_Nm
    cn_beta_per_rad
    engine-out hover: battery end voltage_V
    engine-out hover: generator torque_Nm
    engine-out hover: generator_gearbox power_input_W
    engine-out hover: turboshaft power_shaft_W
    hot-day hover: heat_exchanger power_heat_W
    hot-day hover: turboshaft power_shaft_W
    hover: gearbox power_input_W
    hover: propulsor power_shaft_W
    rotor_radius_m
    static_margin


How to read it: a constraint in **now binding** started sizing something; **no longer binding** means that requirement now has slack (another one took over);
**still binding** are the drivers that persist. Look for the machine-unit counts in `compare` too: whole units jump in steps (tutorial 05), so a variant can
cross a unit boundary and pick up a step in mass.

## 2. More range

Back to 900 kg, but 600 nm instead of 445 nm:

In [6]:
far = size("range 600 nm", requirements=replace(base.requirements, range_m=600 * 1852.0))
compare(base, far)
weight_changes(base, far)

[range 600 nm: sized in 36 s from 'baseline': 16,681 lb]
                                            baseline            range 600 nm
take-off weight [lb]                          15,134         16,681 (+10.2%)
empty weight [lb]                             11,087          11,904 (+7.4%)
payload [lb]                                   1,984           1,984 (+0.0%)
fuel [lb]                                      2,063          2,793 (+35.4%)
battery [kWh]                                   69.5           85.9 (+23.6%)
wing area [m2]                                 23.24           25.14 (+8.2%)
span [m]                                       11.93           12.40 (+4.0%)
rotor radius [m]                               4.825           5.064 (+5.0%)
disk loading [kg/m2]                            46.9            47.0 (+0.1%)
cruise speed [kt]                                161             166 (+3.5%)
cruise L/D                                      9.05            9.35 (+3.3%)
motor rating per la

Compare it with the payload variant. Range pushes **fuel** hardest (+35 % here), and the structure, rotors and drive that carry it. But the battery grew
too (+24 %), even though the mission is not what sizes it. Why? The battery is sized by the **engine-out hover at take-off mass** (tutorial 07): a heavier
aircraft needs more hover power from one engine plus the pack, so the pack grows with take-off weight, not with mission energy. Confirm that the engine-out
battery voltage is still what sizes it:

In [7]:
check("the engine-out hover still sizes the battery at 600 nm",
      any("engine-out hover" in label and "battery end voltage" in label for label in far.result.binding))
growth_mass = far.result.mass_takeoff_kg / base.result.mass_takeoff_kg - 1
growth_battery = far.result.energy_battery_kWh / base.result.energy_battery_kWh - 1
print(f"take-off mass +{growth_mass:.1%}, battery +{growth_battery:.1%}, fuel +{far.result.mass_fuel_kg / base.result.mass_fuel_kg - 1:.1%}")

PASS  the engine-out hover still sizes the battery at 600 nm
take-off mass +10.2%, battery +23.6%, fuel +35.4%


The battery grows faster than take-off mass because it is sized by **voltage sag**: a pack that must deliver more power at the end of the hover needs
disproportionately more parallel strings (tutorial 07, section 6). The lesson for reading any variant: before assuming *why* a part grew, check which
requirement sizes it.

## 3. Less payload

The other direction, 700 kg:

In [8]:
light = size("payload 700 kg", requirements=replace(base.requirements, mass_payload_kg=700.0))
compare(base, light, heavy)

[payload 700 kg: sized in 37 s from 'baseline': 13,539 lb]
                                            baseline          payload 700 kg        payload 1,100 kg
take-off weight [lb]                          15,134         13,539 (-10.5%)          16,252 (+7.4%)
empty weight [lb]                             11,087          10,047 (-9.4%)          11,682 (+5.4%)
payload [lb]                                   1,984          1,543 (-22.2%)          2,425 (+22.2%)
fuel [lb]                                      2,063           1,949 (-5.5%)           2,145 (+4.0%)
battery [kWh]                                   69.5           53.0 (-23.7%)           81.3 (+17.0%)
wing area [m2]                                 23.24           21.26 (-8.5%)           24.62 (+5.9%)
span [m]                                       11.93           11.41 (-4.4%)           12.27 (+2.9%)
rotor radius [m]                               4.825           4.566 (-5.4%)           4.999 (+3.6%)
disk loading [kg/m2]            

Look at the machine-unit row: going lighter, the generators drop from 3 units to 2. A discrete step like that is why weight against payload is not a perfectly
smooth line.

## 4. Your turn

Size a **1,000 kg, 500 nm** variant and answer the three questions below. Edit the two numbers if you want to try something else, then run the cell.

In [9]:
payload_kg = 1000.0            # <- edit
range_nm = 500.0               # <- edit
mine = size(f"{payload_kg:.0f} kg, {range_nm:.0f} nm",
            requirements=replace(base.requirements, mass_payload_kg=payload_kg, range_m=range_nm * 1852.0))
compare(base, mine)
binding_changes(base, mine)

[1000 kg, 500 nm: sized in 35 s from 'baseline': 16,235 lb]
                                            baseline         1000 kg, 500 nm
take-off weight [lb]                          15,134          16,235 (+7.3%)
empty weight [lb]                             11,087          11,671 (+5.3%)
payload [lb]                                   1,984          2,205 (+11.1%)
fuel [lb]                                      2,063          2,359 (+14.4%)
battery [kWh]                                   69.5           81.2 (+16.7%)
wing area [m2]                                 23.24           24.60 (+5.8%)
span [m]                                       11.93           12.27 (+2.9%)
rotor radius [m]                               4.825           4.997 (+3.5%)
disk loading [kg/m2]                            46.9            46.9 (+0.0%)
cruise speed [kt]                                161             163 (+1.7%)
cruise L/D                                      9.05            9.24 (+2.1%)
motor rating per

**Questions**

1. How much take-off weight did it add, and what is the effective growth factor per kilogram of payload-plus-fuel?
2. Which group grew the most?
3. Did anything new start sizing the aircraft?

<details><summary>Answers (for 1,000 kg and 500 nm)</summary>

Run `weight_changes(base, mine)` for question 2. Typically: take-off weight grows by roughly 2.5 × the added payload plus the extra fuel's own snowball; the
largest growths are fuel, the powertrain groups and the wing; the binding list stays dominated by the engine-out hover (battery voltage, turbogenerators), the
4,000 ft hover (drive) and the rotor radius, with possibly a machine-unit step. If a unit count changed, that shows up as a jump in the motor or generator mass.
</details>

## What you practised

- `replace(base.requirements, field=value)` to make a variant; `size(...)` to solve it from the baseline.
- `compare`, `weight_changes`, `binding_changes`: the *what*, the *where* and the *why*.
- Payload grows everything (growth factor about 2.5); range grows fuel most; the battery follows take-off mass through the engine-out hover, not the mission.

Next: **Workbook 2**, requirements that do and do not size the aircraft, and the engines.

In [10]:
check_summary()

2 of 2 checks passed
